In [ ]:
!pip install diffusers transformers accelerate torch safetensors

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
import os

**Task** **1**

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")


In [ ]:
prompts = [
    "Ultra-detailed futuristic megacity at night, neon holograms, flying vehicles, rain, cinematic lighting, cyberpunk aesthetic, 4k, sharp focus",

    "Photorealistic peaceful mountain valley at sunrise, golden hour light, misty atmosphere, pine trees, high dynamic range, ultra detailed",

    "Friendly humanoid robot studying in a modern classroom, books and holographic screen, soft lighting, realistic textures, depth of field",

    "Hyper-realistic portrait of a medieval warrior wearing detailed armor, battle scars, dramatic lighting, shallow depth of field, 8k realism",

    "Cyberpunk street scene at night with heavy rain, neon reflections on wet pavement, moody atmosphere, cinematic composition, ultra detailed"
]


In [ ]:
output_dir = "synthetic_dataset"
os.makedirs(output_dir, exist_ok=True)

for idx, prompt in enumerate(prompts):
    image = pipe(prompt).images[0]
    image.save(f"{output_dir}/image_{idx+1}.png")

print("Synthetic dataset generated successfully.")


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("synthetic_dataset/image_1.png")
plt.imshow(img)
plt.axis("off")


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("synthetic_dataset/image_2.png")
plt.imshow(img)
plt.axis("off")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("synthetic_dataset/image_3.png")
plt.imshow(img)
plt.axis("off")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("synthetic_dataset/image_4.png")
plt.imshow(img)
plt.axis("off")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("synthetic_dataset/image_5.png")
plt.imshow(img)
plt.axis("off")

**Task** **2**

In [ ]:
BASE_DIR = "synthetic_chest_xray_dataset_task2"
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
dataset_labels = {
    "normal_anatomy": "normal healthy adult lungs with clear lung fields and normal cardiomediastinal silhouette",

    "infectious_patterns": "infectious lung disease such as bacterial or viral pneumonia with patchy infiltrates",

    "lung_opacities": "bilateral lung opacities including ground glass opacities and focal consolidations",

    "pleural_conditions": "pleural effusion or pneumothorax with visible pleural abnormalities",

    "structural_lesions": "lung nodules masses fibrotic changes or interstitial lung disease patterns",

    "cardiac_findings": "cardiomegaly with enlarged cardiac silhouette and pulmonary vascular congestion",

    "medical_devices": "presence of medical devices including endotracheal tube central venous catheter or pacemaker leads",

    "imaging_artifacts": "radiographic imaging artifacts such as motion blur noise grid artifacts or exposure imbalance",

    "view_positioning": "different chest X-ray views including PA AP supine and erect positioning",

    "domain_shift": "appearance variations due to different scanners hospitals imaging protocols and resolutions"
}


In [ ]:
BASE_PROMPT = (
    "Highly realistic diagnostic chest X-ray image showing {}, "
    "true radiology appearance, grayscale only, frontal chest radiograph, "
    "accurate anatomy, clinical quality, sharp contrast, hospital PACS style"
)


In [ ]:
IMAGES_PER_CATEGORY = 5

for category, label in dataset_labels.items():
    category_path = os.path.join(BASE_DIR, category)
    os.makedirs(category_path, exist_ok=True)

    prompt = BASE_PROMPT.format(label)
    print(f"Generating images for {category}")

    for i in range(IMAGES_PER_CATEGORY):
        image = pipe(
            prompt,
            guidance_scale=8.5,
            num_inference_steps=40
        ).images[0]

        image.save(f"{category_path}/{category}_{i+1}.png")

print("TASK–2 synthetic dataset generation completed.")


In [ ]:
sample_image = Image.open(
    f"{BASE_DIR}/normal_anatomy/normal_anatomy_1.png"
)

plt.imshow(sample_image)
plt.axis("off")


**TASK 3**

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
model = models.densenet121(pretrained=True)

num_features = model.classifier.in_features
model.classifier = torch.nn.Linear(num_features, 2)  # Normal vs Abnormal
model.eval()


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
DATASET_DIR = "synthetic_chest_xray_dataset_task2"

In [ ]:
def get_true_label(folder_name):
    if folder_name == "normal_anatomy":
        return 0  # Normal
    else:
        return 1  # Abnormal

In [ ]:
correct = 0
total = 0

print("Image | Ground Truth | Predicted")
print("----------------------------------")

for category in os.listdir(DATASET_DIR):
    category_path = os.path.join(DATASET_DIR, category)

    if not os.path.isdir(category_path):
        continue

    true_label = get_true_label(category)

    for img_name in os.listdir(category_path):
        img_path = os.path.join(category_path, img_name)

        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0)

        with torch.no_grad():
            output = model(input_tensor)
            predicted_label = torch.argmax(output, dim=1).item()

        print(f"{img_name} | {true_label} | {predicted_label}")

        if predicted_label == true_label:
            correct += 1

        total += 1


In [ ]:
accuracy = (correct / total) * 100
print("\nTotal Images:", total)
print("Correct Predictions:", correct)
print("Final Accuracy: {:.2f}%".format(accuracy))
